In [1]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

In [3]:
rng = np.random.default_rng(1)
lam = 4
H = 100000


T = np.cumsum(rng.exponential(1/lam, (H, 40)), axis=1)
NA = (T <= 1).sum(1)
NB = rng.poisson(lam, H)
U = [np.sort(rng.random(n)) for n in NB]

for nm, N in (("gaps", NA), ("unif", NB)):
    print(nm, "mean", N.mean(), "var", N.var())

gaps mean 3.999 var 3.987719
unif mean 3.99678 var 4.0142696316


In [4]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
k = np.arange(15)
ax[0].bar(k-.2, np.bincount(NA, minlength=15)[:15]/H, .4, label="gaps")
ax[0].bar(k+.2, np.bincount(NB, minlength=15)[:15]/H, .4, label="unif")
ax[0].plot(k, stats.poisson.pmf(k, lam), "k.-", label="Poisson(4)"); ax[0].legend()
for r in range(10):
    ax[1].vlines(T[r][T[r] <= 1], r, r+.8); ax[2].vlines(U[r], r, r+.8)
ax[1].set_title("gaps"); ax[2].set_title("Poisson + uniform")
x = np.linspace(0, 3, 300)
ax[3].hist(T[:, 2], 80, density=True); ax[3].plot(x, stats.gamma.pdf(x, 3, scale=1/lam), "k")
ax[3].set_title("T3 vs Gamma(3,4)")
fig.savefig("sim.png", dpi=120)

In [8]:
i = NA == 3
c = (T[i] <= .5).sum(1)
# TV distance vs dt (t=1), log-log slope
ns = np.unique(np.logspace(1, np.log10(2000), 20).astype(int))
tv = []
for n in ns:
    k = np.arange(n+1)
    tv.append(.5*(abs(stats.binom.pmf(k, n, lam/n) - stats.poisson.pmf(k, lam)).sum() + stats.poisson.sf(n, lam)))

plt.figure(); plt.loglog(1/ns, tv, "o-"); plt.xlabel("dt"); plt.ylabel("TV"); plt.savefig("tv.png", dpi=120)